In [ ]:
# module list that subclasses abc.Sequence and is jit-compatible

from collections.abc import Iterable, Sequence, Sized
from tempfile import TemporaryFile
from typing import Final, Self, overload

import torch
from torch import Tensor, jit, nn
from torch.nn import Module


class ModuleSequence[M: Module](Module, Sequence[M]):
    # _modules: Dict[str, Module]  # type: ignore[assignment]
    # module_list: list  # type: ignore[assignment]
    length: Final[int]

    def __init__(self, modules: Iterable[Module] = (), /) -> None:
        Module.__init__(self)

        # mods = list(modules)
        # for i, module in mods:
        #     self.register_module(str(i), module)

        self.module_list = nn.ModuleList(modules)
        self.length = len(self.module_list)

    # @jit.export
    @jit.export
    def __len__(self) -> int:
        return self.length

    @overload
    def __getitem__(self, index: int) -> Module: ...
    @overload
    def __getitem__(self, index: slice) -> Self: ...
    def __getitem__(self, index: range | int) -> Module:
        if isinstance(index, int):
            return self.module_list[index]
        return self.__class__(self.module_list[index])

    @jit.export
    def forward(self, x: Tensor) -> Tensor:
        for module in self.module_list:
            x = module(x)
        return x

In [ ]:
example_modules = [
    nn.Linear(3, 3),
    nn.ReLU(),
    nn.Linear(3, 3),
    nn.ReLU(),
]
module_list = ModuleSequence(example_modules)
assert isinstance(module_list.module_list, Iterable)
assert isinstance(module_list.module_list, Sized)
module_list(torch.randn(5, 3))

In [ ]:
scripted = jit.script(module_list)
# len(scripted)

In [ ]:
module_list

In [ ]:
with TemporaryFile() as f:
    jit.save(scripted, f)
    f.seek(0)
    reloaded = jit.load(f)

reloaded(torch.randn(5, 3))